<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/Analytics-Forecast/blob/main/Branch_TimeSeries_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("/content/ODSummary-CSV-AS-ON-04-06-2026.csv")

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

df = df.set_index("Date")

columns_to_fill = [
    "CountofAccountID",
    "CountofCreditOfficerID",
    "SumofPrincipalOutstanding",
    "SumofInterestOutstanding",
    "SumofTotalPrincipalOverdue",
    "SumofTotalInterestOverdue",
    "SumofTotalPAR",
]

df_filled = (
    df.groupby("BranchID")[columns_to_fill]
    .resample("D")
    .ffill()
    .reset_index()
)

df_filled = df_filled[
    ["BranchID", "Date"]
    + [col for col in df_filled.columns if col not in ["BranchID", "Date"]]
]

In [ ]:
df_filled['Day'] = df_filled['Date'].dt.day
df_filled['Month'] = df_filled['Date'].dt.month
df_filled['Year'] = df_filled['Date'].dt.year
df_filled['DayOfWeek'] = df_filled['Date'].dt.dayofweek

In [ ]:
features_to_lag = [
    'CountofAccountID',
    'CountofCreditOfficerID', 'SumofPrincipalOutstanding',
    'SumofInterestOutstanding', 'SumofTotalPrincipalOverdue',
    'SumofTotalInterestOverdue', 'SumofTotalPAR'
]

lags = [1, 7, 15, 30, 365]

for lag in lags:
    for col in features_to_lag:
        lag_col_name = f"{col}_{lag}day_lag"

        df_filled[lag_col_name] = df_filled.groupby('BranchID')[col].shift(lag)

        df_filled[lag_col_name] = df_filled[lag_col_name].fillna(df_filled[col])

print(df_filled.head())

df_filled.to_csv('ml_ready_branch_data.csv')

   BranchID       Date  CountofAccountID  CountofCreditOfficerID  \
0         1 2024-02-01               314                     314   
1         1 2024-02-02               314                     314   
2         1 2024-02-03               314                     314   
3         1 2024-02-04               314                     314   
4         1 2024-02-05               314                     314   

   SumofPrincipalOutstanding  SumofInterestOutstanding  \
0                    4078796                    405923   
1                    4078796                    405923   
2                    4078796                    405923   
3                    4078796                    405923   
4                    4078796                    405923   

   SumofTotalPrincipalOverdue  SumofTotalInterestOverdue  SumofTotalPAR  \
0                     3496538                     366587        4078796   
1                     3503319                     367046        4078796   
2                

In [ ]:
for col in columns_to_fill:
    grouped_col = df_filled.groupby("BranchID")[col]

    # EMA

    df_filled[f"{col}_ema5"] = (
        grouped_col.ewm(span=5, adjust=False)
        .mean()
        .reset_index(level=0, drop=True)
    )
    df_filled[f"{col}_ema7"] = (
        grouped_col.ewm(span=7, adjust=False)
        .mean()
        .reset_index(level=0, drop=True)
    )

    # Rolling max

    df_filled[f"{col}_rolling_max_5"] = (
        grouped_col.rolling(window=5, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )
    df_filled[f"{col}_rolling_max_7"] = (
        grouped_col.rolling(window=7, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )
    df_filled[f"{col}_rolling_max_14"] = (
        grouped_col.rolling(window=14, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )
    df_filled[f"{col}_rolling_max_30"] = (
        grouped_col.rolling(window=30, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )

In [ ]:
columns_to_fill = [
    "CountofAccountID",
    "CountofCreditOfficerID",
    "SumofPrincipalOutstanding",
    "SumofInterestOutstanding",
    "SumofTotalPrincipalOverdue",
    "SumofTotalInterestOverdue",
    "SumofTotalPAR",
]

df_filled = (
    df.groupby("BranchID")[columns_to_fill]
    .resample("D")
    .ffill()
    .reset_index()
)

# Reorder core columns
df_filled = df_filled[
    ["BranchID", "Date"]
    + [col for col in df_filled.columns if col not in ["BranchID", "Date"]]
]

# 2. CRITICAL: Sort data by Branch and Date before calculating time features
df_filled = df_filled.sort_values(["BranchID", "Date"]).reset_index(drop=True)

# 3. Calendar & Cyclical Features
df_filled["Day"] = df_filled["Date"].dt.day
df_filled["Month"] = df_filled["Date"].dt.month
df_filled["Year"] = df_filled["Date"].dt.year
df_filled["DayOfWeek"] = df_filled["Date"].dt.dayofweek
df_filled["Is_Month_End"] = df_filled["Date"].dt.is_month_end.astype(int)

# Encoding cyclical behavior (Helps ML models understand dec-jan or sun-mon connections)
df_filled["Month_Sin"] = np.sin(2 * np.pi * df_filled["Month"] / 12)
df_filled["Month_Cos"] = np.cos(2 * np.pi * df_filled["Month"] / 12)
df_filled["DayOfWeek_Sin"] = np.sin(2 * np.pi * df_filled["DayOfWeek"] / 7)
df_filled["DayOfWeek_Cos"] = np.cos(2 * np.pi * df_filled["DayOfWeek"] / 7)


# 4. Generate Lag Features Safely (No Data Leakage)
lags = [1, 7, 15, 30, 365]

for lag in lags:
    for col in columns_to_fill:
        lag_col_name = f"{col}_{lag}day_lag"
        # Shift within the branch group
        df_filled[lag_col_name] = df_filled.groupby("BranchID")[col].shift(lag)

        # Backfill NaNs ONLY using historical data from the same branch (no cross-contamination)
        df_filled[lag_col_name] = df_filled.groupby("BranchID")[lag_col_name].bfill()


# 5. Generate EMA and Rolling Max Features
for col in columns_to_fill:
    grouped_col = df_filled.groupby("BranchID")[col]

    # EMA Features
    df_filled[f"{col}_ema5"] = (
        grouped_col.ewm(span=5, adjust=False)
        .mean()
        .reset_index(level=0, drop=True)
    )
    df_filled[f"{col}_ema7"] = (
        grouped_col.ewm(span=7, adjust=False)
        .mean()
        .reset_index(level=0, drop=True)
    )

    # Rolling Max Features
    df_filled[f"{col}_rolling_max_5"] = (
        grouped_col.rolling(window=5, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )
    df_filled[f"{col}_rolling_max_7"] = (
        grouped_col.rolling(window=7, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )
    df_filled[f"{col}_rolling_max_14"] = (
        grouped_col.rolling(window=14, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )
    df_filled[f"{col}_rolling_max_30"] = (
        grouped_col.rolling(window=30, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )

df_filled.to_csv('ml_ready_branch_data.csv', index=False)

   BranchID       Date  CountofAccountID  CountofCreditOfficerID  \
0         1 2024-02-01               314                     314   
1         1 2024-02-02               314                     314   
2         1 2024-02-03               314                     314   
3         1 2024-02-04               314                     314   
4         1 2024-02-05               314                     314   

   SumofPrincipalOutstanding  SumofInterestOutstanding  \
0                    4078796                    405923   
1                    4078796                    405923   
2                    4078796                    405923   
3                    4078796                    405923   
4                    4078796                    405923   

   SumofTotalPrincipalOverdue  SumofTotalInterestOverdue  SumofTotalPAR  Day  \
0                     3496538                     366587        4078796    1   
1                     3503319                     367046        4078796    2   
2 

Lag Data missing datas removed

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('/content/ODSummary-CSV-AS-ON-04-06-2026.csv')
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

df = df.set_index("Date")

# ==========================================
# 1. SETUP & RESAMPLING
# ==========================================

columns_to_fill = [
    "CountofAccountID",
    "CountofCreditOfficerID",
    "SumofPrincipalOutstanding",
    "SumofInterestOutstanding",
    "SumofTotalPrincipalOverdue",
    "SumofTotalInterestOverdue",
    "SumofTotalPAR",
]

# Group by Branch and resample to daily intervals, forward-filling missing days
df_filled = (
    df.groupby("BranchID")[columns_to_fill].resample("D").ffill().reset_index()
)

# Reorder core columns so BranchID and Date are upfront
df_filled = df_filled[
    ["BranchID", "Date"]
    + [col for col in df_filled.columns if col not in ["BranchID", "Date"]]
]

# CRITICAL: Strict sorting by Branch and Date before calculating sequential features
df_filled = df_filled.sort_values(["BranchID", "Date"]).reset_index(drop=True)


# ==========================================
# 2. CALENDAR & CYCLICAL FEATURES
# ==========================================

df_filled["Day"] = df_filled["Date"].dt.day
df_filled["Month"] = df_filled["Date"].dt.month
df_filled["Year"] = df_filled["Date"].dt.year
df_filled["DayOfWeek"] = df_filled["Date"].dt.dayofweek
df_filled["Is_Month_End"] = df_filled["Date"].dt.is_month_end.astype(int)

# Encoding cyclical behavior (helps models grasp temporal continuity like Dec -> Jan)
df_filled["Month_Sin"] = np.sin(2 * np.pi * df_filled["Month"] / 12)
df_filled["Month_Cos"] = np.cos(2 * np.pi * df_filled["Month"] / 12)
df_filled["DayOfWeek_Sin"] = np.sin(2 * np.pi * df_filled["DayOfWeek"] / 7)
df_filled["DayOfWeek_Cos"] = np.cos(2 * np.pi * df_filled["DayOfWeek"] / 7)


# ==========================================
# 3. GENERATE LAG FEATURES (LEAK-FREE)
# ==========================================

# NOTE: If your historical data is shorter than a year, consider removing 365
lags = [1, 7, 15, 30, 365]

for lag in lags:
    for col in columns_to_fill:
        lag_col_name = f"{col}_{lag}day_lag"

        # Shift exclusively within the branch group to preserve history timeline
        df_filled[lag_col_name] = df_filled.groupby("BranchID")[col].shift(lag)

        # FIXED: The .bfill() operation was removed from here.
        # Boundary NaNs at the start of the timeline will remain untouched for now.


# ==========================================
# 4. GENERATE EMA & ROLLING METRICS
# ==========================================

for col in columns_to_fill:
    grouped_col = df_filled.groupby("BranchID")[col]

    # Exponential Moving Averages (EMA)
    df_filled[f"{col}_ema5"] = (
        grouped_col.ewm(span=5, adjust=False)
        .mean()
        .reset_index(level=0, drop=True)
    )
    df_filled[f"{col}_ema7"] = (
        grouped_col.ewm(span=7, adjust=False)
        .mean()
        .reset_index(level=0, drop=True)
    )

    # Rolling Maximums (min_periods=1 allows calculation even with partial windows)
    df_filled[f"{col}_rolling_max_5"] = (
        grouped_col.rolling(window=5, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )
    df_filled[f"{col}_rolling_max_7"] = (
        grouped_col.rolling(window=7, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )
    df_filled[f"{col}_rolling_max_14"] = (
        grouped_col.rolling(window=14, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )
    df_filled[f"{col}_rolling_max_30"] = (
        grouped_col.rolling(window=30, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )


# ==========================================
# 5. SAFE BOUNDARY CLEANUP & EXPORT
# ==========================================

# Dynamically find out what your maximum lag actually is
max_lag_used = max(lags)

# Drop only the rows that don't have enough history to fulfill the maximum lag.
# This cleanly discards historical initialization gaps without contaminating past rows.
lag_check_column = f"SumofTotalInterestOverdue_{max_lag_used}day_lag"
df_filled = df_filled.dropna(subset=[lag_check_column]).reset_index(drop=True)

# Save the pristine, leak-free dataset
df_filled.to_csv("ml_ready_branch_data.csv", index=False)
print(
    f"Data pipeline complete. Rows dropped due to initial {max_lag_used}-day lag limit."
)